# TabPFN (corre en Kaggle con GPU)

TabPFN es un modelo fundacional para datos tabulares: un transformer preentrenado sobre millones de tablas sintéticas
que, en vez de entrenarse, recibe las filas de entrenamiento como contexto y predice las nuevas en una pasada.

Usa exactamente las mismas features y los mismos 5 folds que LightGBM/XGBoost/CatBoost (dataset de Kaggle
generado con `gpu_kaggle/exportar_datos_tabpfn.py`), así que su predicción es comparable
y se puede sumar al ensamble. Requiere el secret `TABPFN_TOKEN` (licencia de Prior Labs).

**Cómo correrlo:** notebook de Kaggle con GPU e internet, con el dataset exportado como entrada y el secret `TABPFN_TOKEN`. Tarda unas 2 a 3 horas. La salida `tabpfn.npz` va a `gpu_kaggle/salidas/tabpfn.npz` (claves `oof`, `valid0`..`valid4` y `test`).

In [ ]:
!pip install -q tabpfn==9.2.0   # version con la que se genero la salida (modelo TabPFN-3.5)

In [ ]:
import glob, os, time, numpy as np, pandas as pd, torch
from kaggle_secrets import UserSecretsClient
os.environ["TABPFN_TOKEN"] = UserSecretsClient().get_secret("TABPFN_TOKEN")
from tabpfn import TabPFNClassifier
import tabpfn
print("tabpfn", tabpfn.__version__, "| GPU:", torch.cuda.get_device_name(0))

DATA = os.path.dirname(glob.glob("/kaggle/input/**/fold0_tr.parquet", recursive=True)[0])
y = np.load(f"{DATA}/y.npy")
N_ESTIMATORS = 4   # mismo valor que la corrida que entra en ensamble.csv
SEED = 42

In [ ]:
def prepare(A, B):
    """Categoricas -> codigos enteros con el mismo diccionario en A y B."""
    A, B = A.copy(), B.copy()
    cats = [c for c in A.columns if str(A[c].dtype) == "category"]
    for c in cats:
        dic = pd.Index(pd.concat([A[c].astype(str), B[c].astype(str)]).unique())
        A[c], B[c] = dic.get_indexer(A[c].astype(str)), dic.get_indexer(B[c].astype(str))
    return A.astype("float32"), B.astype("float32"), [A.columns.get_loc(c) for c in cats]

def fit_predict(A, ya, B):
    A, B, cat_idx = prepare(A, B)
    m = TabPFNClassifier(n_estimators=N_ESTIMATORS, categorical_features_indices=cat_idx, device="cuda",
                         ignore_pretraining_limits=True, random_state=SEED)
    m.fit(A.values, ya)
    return np.concatenate([m.predict_proba(B.values[s:s + 2000])[:, 1] for s in range(0, len(B), 2000)])

In [ ]:
out = {"oof": np.zeros(len(y), dtype="float32")}
t0 = time.time()
from sklearn.model_selection import StratifiedKFold
for f, (a, b) in enumerate(StratifiedKFold(5, shuffle=True, random_state=0).split(y, y)):   # = features.cv_folds
    A, B = pd.read_parquet(f"{DATA}/fold{f}_tr.parquet"), pd.read_parquet(f"{DATA}/fold{f}_va.parquet")
    p = fit_predict(A, y[a], B)
    out["oof"][b] = p; out[f"valid{f}"] = p
    print(f"fold {f}: accuracy {((p > .5) == y[b]).mean():.4f}  ({(time.time() - t0) / 60:.0f} min)", flush=True)
    np.savez("tabpfn.npz", **out)
print(f"Accuracy CV TabPFN: {((out['oof'] > .5) == y).mean():.4f}")

In [ ]:
A, B = pd.read_parquet(f"{DATA}/full_tr.parquet"), pd.read_parquet(f"{DATA}/full_te.parquet")
out["test"] = fit_predict(A, y, B)
np.savez("tabpfn.npz", **out)
print(f"test listo ({(time.time() - t0) / 60:.0f} min)")